# DeepTuneIO — Module 06: Darshan Performance Processor

Procesa exclusivamente los reportes `processed/Perf/*.txt`.

Este módulo **no consolida Parser ni SEFF**. Esa integración corresponde al
Module 09.

Las métricas `agg_time_by_slowest` y `agg_perf_by_slowest` se conservan como
valores proporcionados directamente por Darshan. Las métricas calculadas por
DeepTuneIO se mantienen identificadas por separado.


**Procedencia de nodos:** Module 06 no genera `Nodes`, `Processes_per_Node` ni `Bytes_per_Node`, porque Perf no aporta directamente el número de nodos. Module 09 incorporará `Nodes` desde el Summary Global de DXT mediante `JobID` y calculará allí las métricas dependientes del número de nodos.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE06_FILE = MODULE_DIR / "06_process_performance_reports.py"

if not MODULE06_FILE.exists():
    raise FileNotFoundError(f"No se encontró Module 06: {MODULE06_FILE}")

spec = spec_from_file_location("deeptuneio_module06", MODULE06_FILE)
module06 = module_from_spec(spec)
sys.modules[spec.name] = module06
spec.loader.exec_module(module06)

print("Module 08:", MODULE06_FILE)
print("Version  :", module06.VERSION)


Module 08: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\06_Performance\06_Performance_Processing\Module\06_process_performance_reports.py
Version  : 1.1.0


## 1. Seleccionar campaña por índice


In [2]:
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": [
                "lustre_1ost",
                "lustre_2ost",
                "lustre_4ost",
                "nfs_SnGPU_e1",
            ],
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"]},
        "NPZ_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"]},
    },
}

APPLICATION_POS = 2
DATASET_POS = 4
SCENARIO_POS = 1

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]

print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)


Application   : DLIOv1
Dataset group : TFRecord_1mts_64bs
Scenario      : lustre_ost


## 2. Construir la raíz de campaña


In [3]:
PROJECT_ROOT = Path(r"F:\NoteBooks\CORA_DR_UAB").resolve()
ARTICLE_ID = "Article_01_Parallel_IO_Analysis"

ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID
CAMPAIGN_ROOT = ARTICLE_ROOT / "Data" / APPLICATION / DATASET_GROUP / SCENARIO

print("Project root  :", PROJECT_ROOT)
print("Article root  :", ARTICLE_ROOT)
print("Campaign root :", CAMPAIGN_ROOT)
print("Exists        :", CAMPAIGN_ROOT.exists())


Project root  : F:\NoteBooks\CORA_DR_UAB
Article root  : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\lustre_ost
Exists        : True


## 3. Resolver rutas


In [4]:
args = module06.parse_args([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", "JSC24",
    "--module-root", str(MODULE_DIR),
])

paths = module06.resolve_campaign_paths(args)

for key, value in paths.items():
    print(f"{key:14s}: {value}")

if paths["perf_dir"].exists():
    print("Perf TXT      :", len(list(paths["perf_dir"].glob("*.txt"))))


campaign_root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost
module_root   : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\06_Performance\06_Performance_Processing\Module
perf_dir      : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\processed\Perf
output_dir    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Perf
Perf TXT      : 13


## 4. Ejecutar Module 06


In [5]:
return_code = module06.main([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", "JSC24",
    "--module-root", str(MODULE_DIR),
])

print("Return code:", return_code)


DeepTuneIO — Module 06: Darshan Performance Processor
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost
Application   : DLIOv1
Dataset group : TFRecord_1mts_64bs
Scenario      : lustre_ost
Perf input    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\processed\Perf
Perf TXT      : 13
Perf output   : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Perf\08_Perf_DLIOv1_lustre_ost_TFRecord_1mts_64bs_JSC24.csv


Procesando Perf: 100%|██████████| 13/13 [00:00<00:00, 49.69it/s]

Rows generated: 13
CSV saved     : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Perf\08_Perf_DLIOv1_lustre_ost_TFRecord_1mts_64bs_JSC24.csv
Return code: 0


## 5. Revisar salida Perf


In [6]:
PERF_RESULTS_DIR = CAMPAIGN_ROOT / "results" / "Perf"
generated = sorted(PERF_RESULTS_DIR.glob("06_Perf_*.csv"))

print("Perf results:", PERF_RESULTS_DIR)
for p in generated:
    print(" -", p.name)

if generated:
    perf_df = pd.read_csv(generated[-1])
    display(perf_df.head())
    print("Rows:", len(perf_df))


Perf results: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\lustre_ost\results\Perf


## 6. Siguiente etapa

```text
Module 05 Parser ─┐
Module 06 Perf   ─┼──► Module 09 Consolidation
Module 03 SEFF   ─┘
```

La consolidación debe realizarse por `JobID` y validar la coherencia de campaña,
escenario y procesos antes de unir las fuentes.
